<a href="https://colab.research.google.com/github/melaniedaniel7/CFPB-Text-Analysis-Using-NLP-Techniques/blob/main/Text_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Text Cleaning**

## **Objectives:**



* Import the dataset
* Sort the data for efficient grouping
* Clean the text with example
* Clean the entire dataset



# **1: Import the dataset**

In [34]:
# Imported the dataset
from google.colab import files

uploaded = files.upload()

Saving complaints-2026-06-17_10_50.csv to complaints-2026-06-17_10_50.csv


In [35]:
# Checked the dataset uploaded correctly
import pandas as pd

df = pd.read_csv("complaints-2026-06-17_10_50.csv")
df.head()
df.columns

Index(['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue',
       'Consumer complaint narrative', 'Company public response', 'Company',
       'State', 'ZIP code', 'Tags', 'Submitted via', 'Date sent to company',
       'Company response to consumer', 'Timely response?', 'Complaint ID'],
      dtype='object')

In [36]:
# Checked if any complaint narratives were missing
df.shape
df['Consumer complaint narrative'].isnull().sum()

np.int64(0)

# **2: Sort the data for efficient grouping**

In [48]:
# Sort the data

# Sorting by product and subproduct
# From filtering the dataset on the CFPB website, I already knew there were 4 product categories
# This code was created with the help of Google Gemini, which is built into Google Colab
grouped_products = df.groupby('Product')['Sub-product'].agg(['nunique', lambda x: x.unique().tolist()])
grouped_products.columns = ['Number of Sub-products', 'List of Sub-products']

# Reset index to make 'Product' a column and add 'Product number'
grouped_products = grouped_products.reset_index()
grouped_products.insert(0, 'Product number', range(1, 1 + len(grouped_products)))

display(grouped_products)

,Product number,Product,Number of Sub-products,List of Sub-products
0,1,Credit card,2,"[General-purpose credit card or charge card, S..."
1,2,Debt collection,11,"[I do not know, Rental debt, Telecommunication..."
2,3,Mortgage,8,"[FHA mortgage, Conventional home mortgage, VA ..."
3,4,Student loan,2,"[Federal student loan servicing, Private stude..."


In [54]:
# Seeing how many consumer complaints there are for each product and sub-product
# This code was created with the help of Google Gemini, which is built into Google Colab
complaints_per_product_subproduct = df.groupby(['Product', 'Sub-product']).size().reset_index(name='Number of Complaints')
display(complaints_per_product_subproduct)

,Product,Sub-product,Number of Complaints
0,Credit card,General-purpose credit card or charge card,3884
1,Credit card,Store credit card,577
2,Debt collection,Auto debt,261
3,Debt collection,Credit card debt,1360
4,Debt collection,Federal student loan debt,23
5,Debt collection,I do not know,3342
6,Debt collection,Medical debt,330
7,Debt collection,Mortgage debt,26
8,Debt collection,Other debt,1325
9,Debt collection,Payday loan debt,110


From these results, I decided to prioritise the dataset and consumer complaints by picking a single subcategory for each product with the highest number of complaints for topic extraction (identifying the most prevalent topics).

I have decidied to do this to ensure that the identified topics are as accurate and reliable as possible.

This is because in phase two I identified the most prevalent topics across the corpus but as one can see from the table above this would most likely not be an accurate reflection of the data and the consumer comlaints.

For example, within the corpus, there are a total of 4461 consumer complaints for the product "Credit card," but 759 consumer complaints for the product "Student loan".

Therefore, I will only be working with the following data moving forward:

1) Credit card (General-purpose credit card or charge card) = 3884 complaints
2) Debt collection (I do not know) = 3342 complaints
3) Mortgage (Conventional home mortgage)= 866 complaints
4) Student loan (Federal student loan servicing) = 594 complaints

In [50]:
# Sort the data

# Sorting the prioritised data by issue and sub-issue
grouped_issues = df.groupby('Issue')['Sub-issue'].agg(['nunique', lambda x: x.unique().tolist()])
grouped_issues.columns = ['Number of Sub-issues', 'List of Sub-issues']
print(f"Total Number of Issues: {len(grouped_issues)}")

grouped_issues = grouped_issues.reset_index()
grouped_issues.insert(0, 'Issue number', range(1, 1 + len(grouped_issues)))

display(grouped_issues)

Total Number of Issues: 30


,Issue number,Issue,Number of Sub-issues,List of Sub-issues
0,1,"Advertising and marketing, including promotion...",2,[Confusing or misleading advertising about the...
1,2,Applying for a mortgage or refinancing an exis...,8,"[Application denials, Fees or costs during the..."
2,3,Attempts to collect debt not owed,4,"[Debt was result of identity theft, Debt is no..."
3,4,Closing on a mortgage,6,[Closing disclosure or other related disclosur...
4,5,Closing your account,2,"[Company closed your account, Can't close your..."
5,6,Communication tactics,4,"[You told them to stop contacting you, but the..."
6,7,Credit monitoring or identity theft protection...,5,"[Billing dispute for services, Didn't receive ..."
7,8,Dealing with your lender or servicer,7,"[Problem with customer service, Trouble with h..."
8,9,Electronic communications,4,"[Frequent or repeated messages, Contacted befo..."
9,10,False statements or representation,4,"[Attempted to collect wrong amount, Told you n..."


In [42]:
grouped_products = df.groupby('Product')['Sub-product'].agg(['nunique', lambda x: x.unique().tolist()])
grouped_products.columns = ['Number of Sub-products', 'List of Sub-products']
display(grouped_products)

,Number of Sub-products,List of Sub-products
Product,,
Credit card,2,"[General-purpose credit card or charge card, S..."
Debt collection,11,"[I do not know, Rental debt, Telecommunication..."
Mortgage,8,"[FHA mortgage, Conventional home mortgage, VA ..."
Student loan,2,"[Federal student loan servicing, Private stude..."


In [ ]:
# Dataset only contains complaint narratives now
complaints = df[['Consumer complaint narrative']].copy()
complaints = complaints.dropna()
complaints.shape

In [52]:
# Viewed an example complaint
# Realised that I have to clean the redacted text, represented by "XXX's", when doing text cleaning
# The 'complaints' DataFrame was not defined when this cell was executed.
# It is defined here to ensure the cell runs correctly.
complaints = df[['Consumer complaint narrative']].copy()
complaints = complaints.dropna()
complaints.head()
print(complaints['Consumer complaint narrative'].iloc[0])

I was told by XXXX XXXX that XXXX XXXX XXXX  gave them information about the status of my account and whether it was paid off or not and how much I owed. I didnt authorize that information to be given to any one.


In [ ]:
# Import libraries and download NLTK resources for text cleaning
import nltk
# These libraries were not mentioned in phase 1 and were an additional add-on for efficient text preprocessing
import re
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag # Import for POS tagging

# Punkt tokenizer for word tokenization
nltk.download('punkt')
# Stopwords list in multiple languages
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
nltk.download('wordnet') # Added to resolve LookupError for WordNetLemmatizer
nltk.download('averaged_perceptron_tagger') # Added for POS tagging
nltk.download('averaged_perceptron_tagger_eng') # Added to resolve LookupError for averaged_perceptron_tagger_eng

# Create a set of English stopwords for efficient lookup
stop_words = set(stopwords.words('english'))

# Lemmatizer converts words to their base/root form
lemmatizer = WordNetLemmatizer()

# Function to convert NLTK POS tags to WordNet POS tags
def get_wordnet_pos(treebank_tag):
    if treebank_tag.startswith('J'):
        return nltk.corpus.wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return nltk.corpus.wordnet.VERB
    elif treebank_tag.startswith('N'):
        return nltk.corpus.wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return nltk.corpus.wordnet.ADV
    else:
        return nltk.corpus.wordnet.NOUN # Default to noun if not found

# Function to clean the text by converting text to lowercase and removing punctuation
def clean_text(text):
    if isinstance(text, str):
        text = text.lower()
        # Remove patterns like 'xxxx' and 'xxxxyear'
        text = re.sub(r'x{2,}', '', text) # This will remove 'xxxx' and 'xxxxyear' will become 'year'
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text)
        return text
    else:
        return text

# Function to tokenize the text into individual words
def tokenize_text(text):
    if isinstance(text, str):
        tokens = word_tokenize(text)
        # The previous filter for set(word.lower()) == {'x'} is now less critical
        # as most 'x' sequences are handled by clean_text, but can keep it as a safeguard.
        tokens = [
            word for word in tokens
            if word and not (len(word) == 1 and word.lower() == 'x') # Exclude single 'x' if it appears
        ]
        return tokens
    else:
        return []

# Function to remove stopwords from the tokenized text
def remove_stopwords(tokens):
    if isinstance(tokens, list):
        return [word for word in tokens if word not in stop_words]
    else:
        return tokens

# Function to apply lemmatization to the tokens with POS tagging
def lemmatize_tokens(tokens):
    if isinstance(tokens, list):
        lemmatized_words = []
        for word, tag in pos_tag(tokens):
            wntag = get_wordnet_pos(tag)
            lemmatized_words.append(lemmatizer.lemmatize(word, wntag))
        return lemmatized_words
    else:
        return tokens

# Test text cleaning techniques on a single consumer complaint before applying text cleaning to the entire dataset
# Used the same complaint example as earlier
sample = df['Consumer complaint narrative'].iloc[0]
print("Original:")
print(sample)
# Clean example complaint
cleaned = clean_text(sample)
print("\nCleaned:")
print(cleaned)
# Tokenize example complaint
tokens = tokenize_text(cleaned)
print("\nTokens:")
print(tokens)
# Remove stopwords from example complaint
filtered = remove_stopwords(tokens)
print("\nNO Stpwords:")
print(filtered)
# Lemmatize example complaint
final = lemmatize_tokens(filtered)
print("\nLemmatized:")
print(final)

In [ ]:
# Apply the functions to the DataFrame
# Clean the text
df['Cleaned_Complaint'] = df['Consumer complaint narrative'].apply(clean_text)
# Tokenize the cleaned text
df['Tokenized_Complaint'] = df['Cleaned_Complaint'].apply(tokenize_text)
# Remove stopwords from the tokenized text
df['No_Stopwords_Complaint'] = df['Tokenized_Complaint'].apply(remove_stopwords)
# Apply lemmatization to the tokenized words
df['Lemmatized_Complaint'] = df['No_Stopwords_Complaint'].apply(lemmatize_tokens)

# Display the original and lemmatized complaint narratives
display(df[['Consumer complaint narrative', 'Lemmatized_Complaint']].head())